# Audio-to-Frame Mapping

Connects audio segments to visual frames for a chosen debate.

At 1 fps, frame N covers the time window **[N-1, N) seconds**, so:
- `frame_start = floor(time_stamp) + 1`
- `frame_end   = ceil(time_stamp + duration)` (last frame with any overlap)

**Change `DEBATE` below** to run on any debate video.

In [2]:
DEBATE        = 'Martins_vs_Gouveia_Melo_November_23'
FEATURES_DIR  = '../Project_Features'

In [3]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

## 1 — Load audio data and assign speaker labels

Speaker labels require the cross-debate pipeline: all 28 debates are loaded,
k=3 clusters are built per debate, global per-candidate centroids are derived,
then clusters are matched to candidate names. Condensed from `audio_candidate_analysis.ipynb`.

In [4]:
audio_files = sorted([f for f in os.listdir(FEATURES_DIR) if f.endswith('_audio.pkl')])

dfs = []
for f in audio_files:
    df = pd.read_pickle(os.path.join(FEATURES_DIR, f))
    df['video'] = f.replace('_audio.pkl', '')
    dfs.append(df)

data_audio = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(data_audio)} segments across {data_audio["video"].nunique()} videos')

Loaded 2656 segments across 28 videos


In [5]:
def parse_embedding(s):
    if isinstance(s, np.ndarray):
        return s
    return np.fromstring(s.strip('[]'), sep=' ')

data_audio['speak_embeddings'] = data_audio['speak_embeddings'].apply(parse_embedding)

data_parties       = pd.read_pickle(os.path.join(FEATURES_DIR, 'candidate_party.pkl'))
candidate_to_party = dict(zip(data_parties['Candidate'], data_parties['Party']))

CANDIDATES = [
    'Cotrim_Figueiredo', 'Filipe', 'Gouveia_Melo',
    'Marques_Mendes', 'Martins', 'Pinto', 'Seguro', 'Ventura'
]
debate_videos = sorted([v for v in data_audio['video'].unique() if 'vs' in v])

def extract_candidates(video_name):
    for c in CANDIDATES:
        if video_name.startswith(c):
            rest = video_name[len(c) + len('_vs_'):]
            for c2 in CANDIDATES:
                if rest.startswith(c2):
                    return c, c2
    return None, None

In [6]:
debate_centroids = {}
cluster_col      = np.full(len(data_audio), -1, dtype=int)

for video_name in debate_videos:
    mask = data_audio['video'] == video_name
    idx  = data_audio.index[mask]
    E    = np.stack(data_audio.loc[idx, 'speak_embeddings'].values)
    km   = KMeans(n_clusters=3, random_state=42, n_init=10)
    lbls = km.fit_predict(E)
    cluster_col[idx]              = lbls
    debate_centroids[video_name]  = km.cluster_centers_

data_audio['cluster_k3'] = cluster_col
print(f'k=3 clustering done for {len(debate_centroids)} debates')

k=3 clustering done for 28 debates


In [7]:
def candidate_centroid_per_debate(candidate, debate_centroids):
    their_debates = [v for v in debate_centroids if candidate in v]
    if len(their_debates) < 2:
        return {}
    best = {}
    for video in their_debates:
        others = [v for v in their_debates if v != video]
        cents  = debate_centroids[video]
        scores = [
            sum(cdist(cents[i].reshape(1,-1), debate_centroids[ov], metric='cosine')[0].min()
                for ov in others)
            for i in range(3)
        ]
        best[video] = cents[int(np.argmin(scores))]
    return best

global_centroids = {}
for cand in CANDIDATES:
    result = candidate_centroid_per_debate(cand, debate_centroids)
    if result:
        global_centroids[cand] = np.mean(np.stack(list(result.values())), axis=0)

print(f'Global centroids built for {len(global_centroids)} candidates')

Global centroids built for 8 candidates


In [8]:
speaker_col = np.full(len(data_audio), 'unknown', dtype=object)

for video_name in debate_videos:
    cA, cB = extract_candidates(video_name)
    if cA is None:
        continue
    mask  = data_audio['video'] == video_name
    idx   = data_audio.index[mask]
    cents = debate_centroids[video_name]

    dist_A = cdist(cents, global_centroids[cA].reshape(1,-1), metric='cosine').flatten()
    dist_B = cdist(cents, global_centroids[cB].reshape(1,-1), metric='cosine').flatten()

    cluster_map = {}
    available   = {0, 1, 2}
    best_A = int(np.argmin([dist_A[i] if i in available else np.inf for i in range(3)]))
    cluster_map[best_A] = cA
    available.remove(best_A)
    best_B = int(np.argmin([dist_B[i] if i in available else np.inf for i in range(3)]))
    cluster_map[best_B] = cB
    available.remove(best_B)
    cluster_map[available.pop()] = 'host'

    labels = data_audio.loc[idx, 'cluster_k3'].values
    speaker_col[idx] = [cluster_map[l] for l in labels]

data_audio['speaker'] = speaker_col
data_audio['party']   = data_audio['speaker'].map(candidate_to_party).fillna('host')

print('Speaker assignment done.')
print(data_audio[data_audio['video'] == DEBATE]['speaker'].value_counts().to_string())

Speaker assignment done.
speaker
Gouveia_Melo    57
Martins         26
host            16


## 2 — Load visual data for the chosen debate

In [9]:
visual_file = os.path.join(FEATURES_DIR, f'{DEBATE}_visual.pkl')
data_visual = pd.read_pickle(visual_file)

data_visual['frame_number'] = (
    data_visual['Frame']
    .str.extract(r'frame_(\d+)\.jpg')[0]
    .astype(int)
)
data_visual = data_visual.sort_values('frame_number').reset_index(drop=True)

def count_faces(face_list):
    if isinstance(face_list, list):
        return len(face_list)
    return 0

data_visual['face_count'] = data_visual['Fer'].apply(count_faces)

total_frames = len(data_visual)
print(f'Debate  : {DEBATE}')
print(f'Frames  : {total_frames}  (range {data_visual["frame_number"].min()}–{data_visual["frame_number"].max()})')
print(f'Face count distribution:')
print(data_visual['face_count'].value_counts().sort_index().to_string())

Debate  : Martins_vs_Gouveia_Melo_November_23
Frames  : 2048  (range 1–2048)
Face count distribution:
face_count
0      1
1    989
2    901
3    156
4      1


## 3 — Build audio-to-frame mapping

In [10]:
debate_audio = (
    data_audio[data_audio['video'] == DEBATE]
    .copy()
    .sort_values('time stamp')
    .reset_index(drop=True)
)

debate_audio['frame_start'] = (debate_audio['time stamp'].apply(math.floor) + 1).astype(int)
debate_audio['frame_end']   = (debate_audio['time stamp'] + debate_audio['duration']).apply(math.ceil).astype(int)
debate_audio['n_frames']    = debate_audio['frame_end'] - debate_audio['frame_start'] + 1
debate_audio['frames']      = debate_audio.apply(
    lambda r: list(range(r['frame_start'], r['frame_end'] + 1)), axis=1
)

print(f'Mapping built for {len(debate_audio)} audio segments.')

Mapping built for 99 audio segments.


## 4 — Summary statistics

In [11]:
all_audio_frames   = set(f for frames in debate_audio['frames'] for f in frames)
existing_frames    = set(data_visual['frame_number'].unique())
covered_frames     = all_audio_frames & existing_frames
not_covered_frames = existing_frames - all_audio_frames

speaker_stats = (
    debate_audio
    .groupby(['speaker', 'party'])
    .agg(
        segments         = ('time stamp', 'count'),
        total_duration_s = ('duration', 'sum'),
        total_frames     = ('n_frames', 'sum')
    )
    .reset_index()
    .sort_values('segments', ascending=False)
)

print('=' * 58)
print(f'  Debate : {DEBATE}')
print('=' * 58)
print(f'  Audio segments            : {len(debate_audio)}')
print(f'  Total frames (visual)     : {total_frames}')
print(f'  Frames covered by audio   : {len(covered_frames)}  ({len(covered_frames)/total_frames*100:.1f}%)')
print(f'  Frames NOT in any segment : {len(not_covered_frames)}  ({len(not_covered_frames)/total_frames*100:.1f}%)')
print()
print('Per-speaker breakdown:')
print(speaker_stats.to_string(index=False))

  Debate : Martins_vs_Gouveia_Melo_November_23
  Audio segments            : 99
  Total frames (visual)     : 2048
  Frames covered by audio   : 1795  (87.6%)
  Frames NOT in any segment : 253  (12.4%)

Per-speaker breakdown:
     speaker       party  segments  total_duration_s  total_frames
Gouveia_Melo Independent        57           647.696           705
     Martins          BE        26           882.731           912
        host        host        16           210.449           225


## 5 — Interleaved timeline: audio segments + gaps

Gap rows (yellow) show the frames between two consecutive audio segments,
how many frames they span, and the duration of the silence in seconds.

In [12]:
rows = []
for i in range(len(debate_audio)):
    seg = debate_audio.iloc[i]

    # ── gap before this segment ────────────────────────────────────────────
    if i > 0:
        prev      = debate_audio.iloc[i - 1]
        g_f_start = int(prev["frame_end"]) + 1
        g_f_end   = int(seg["frame_start"]) - 1
        if g_f_start <= g_f_end:
            gap_s = seg["time stamp"] - (prev["time stamp"] + prev["duration"])
            rows.append({
                "type"       : "gap",
                "segment"    : "",
                "t_start_s"  : "",
                "duration_s" : f"{gap_s:.2f}",
                "speaker"    : "(silence)",
                "party"      : "",
                "frame_start": g_f_start,
                "frame_end"  : g_f_end,
                "n_frames"   : g_f_end - g_f_start + 1,
            })

    # ── audio segment ─────────────────────────────────────────────────────
    rows.append({
        "type"       : "audio",
        "segment"    : i,
        "t_start_s"  : f"{seg['time stamp']:.3f}",
        "duration_s" : f"{seg['duration']:.3f}",
        "speaker"    : seg["speaker"],
        "party"      : seg["party"],
        "frame_start": int(seg["frame_start"]),
        "frame_end"  : int(seg["frame_end"]),
        "n_frames"   : int(seg["n_frames"]),
    })

timeline_df = pd.DataFrame(rows).reset_index(drop=True)
display_df  = timeline_df.drop(columns="type")

# _style_row gets a row from display_df (no type col); look it up in timeline_df by index
def _style_row(row):
    if timeline_df.loc[row.name, "type"] == "gap":
        return ["background-color: #fff8dc; color: #7a6000"] * len(row)
    return [""] * len(row)

(
    display_df
    .style
    .apply(_style_row, axis=1)
    .set_caption(f"Timeline — {DEBATE}  |  audio segments (white) + silence gaps (yellow)")
)

,segment,t_start_s,duration_s,speaker,party,frame_start,frame_end,n_frames
0,0,0.031,23.726,host,host,1,24,24
1,,,1.32,(silence),,25,25,1
2,1,25.073,16.048,Gouveia_Melo,Independent,26,42,17
3,2,42.455,9.551,Gouveia_Melo,Independent,43,53,11
4,3,53.187,6.902,Gouveia_Melo,Independent,54,61,8
5,4,60.764,18.411,Gouveia_Melo,Independent,61,80,20
6,5,79.647,16.132,Gouveia_Melo,Independent,80,96,17
7,6,95.780,13.888,host,host,96,110,15
8,7,110.444,6.699,Gouveia_Melo,Independent,111,118,8
9,,,3.61,(silence),,119,120,2


## 6 — Frame slideshow: top 3 segments per speaker

For each speaker (both candidates + host), shows the **3 longest audio segments**.
Every visual frame within the segment is shown — not just single-face ones.
Use the Play button or slider to scrub through and see what the camera was capturing.

In [14]:
import ipywidgets as widgets
from IPython.display import display as ipy_display, Audio as IPyAudio
import threading, base64 as _b64, wave as _wave_lib
import time as _time
from io import BytesIO as _BytesIO
from PIL import ImageDraw as _Draw
import scipy.io.wavfile as _wavfile

FRAMES_DIR = '../Frames'
AUDIO_DIR  = '../Audio'

def resolve_frame_path(pkl_path, frame_number):
    debate_folder = os.path.basename(os.path.dirname(pkl_path))
    if frame_number < 1000:
        return os.path.join(FRAMES_DIR, debate_folder, f'frame_{frame_number:03d}.jpg')
    return os.path.join(FRAMES_DIR, debate_folder, f'frame_{frame_number}.jpg')

# Load WAV once
_wav_path = os.path.join(AUDIO_DIR, f'{DEBATE}.wav')
if os.path.exists(_wav_path):
    _sr, _full_audio = _wavfile.read(_wav_path)
    if _full_audio.ndim == 2:
        _full_audio = _full_audio[:, 0]
    _full_audio = _full_audio.astype(np.float32) / 32768.0
    print(f'WAV loaded: sr={_sr} Hz, duration={len(_full_audio)//_sr}s')
else:
    _sr = None; _full_audio = None
    print(f'[!] WAV not found: {_wav_path}')

_resample = getattr(Image, 'Resampling', Image).LANCZOS

def _make_audio_html(chunk, sr):
    pcm = (chunk * 32767).astype(np.int16)
    buf = _BytesIO()
    with _wave_lib.open(buf, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())
    b64 = _b64.b64encode(buf.getvalue()).decode()
    return widgets.HTML(
        f'<audio controls style="width:900px" '
        f'src="data:audio/wav;base64,{b64}"></audio>'
    )

def _frame_bytes(frame_num, vis_row, target_w=900):
    img_path = resolve_frame_path(vis_row['Frame'], frame_num)
    try:
        img = Image.open(img_path).convert('RGB')
    except (FileNotFoundError, OSError):
        img = Image.new('RGB', (target_w, 506), (30, 30, 30))
    orig_w, orig_h = img.size
    target_h = int(orig_h * target_w / orig_w)
    sx, sy = target_w / orig_w, target_h / orig_h
    img  = img.resize((target_w, target_h), _resample)
    draw = _Draw.Draw(img)
    for fer in vis_row['Fer']:
        if not isinstance(fer, dict):
            continue
        x1, y1, x2, y2 = fer['bbox']
        draw.rectangle([int(x1*sx), int(y1*sy), int(x2*sx), int(y2*sy)],
                       outline='lime', width=3)
        draw.text((int(x1*sx)+2, max(int(y1*sy)-18, 0)),
                  fer.get('top_emotion', ''), fill='lime')
    buf = _BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

def _make_block(seg, all_frames, rank, speaker, frame_lookup):
    vrows = {f: frame_lookup.loc[f] for f in all_frames}

    header = widgets.HTML(
        f'<b>{speaker}</b> &nbsp;|&nbsp; {seg["party"]} &nbsp;|&nbsp; '
        f'Rank #{rank} by duration &nbsp;|&nbsp; '
        f't={seg["time stamp"]:.1f}s &nbsp; dur={seg["duration"]:.1f}s &nbsp;|&nbsp; '
        f'{len(all_frames)} frames'
    )
    img_w    = widgets.Image(format='jpeg', layout=widgets.Layout(width='900px'))
    slider   = widgets.IntSlider(value=0, min=0, max=len(all_frames)-1,
                                  description='Frame:', continuous_update=True,
                                  layout=widgets.Layout(width='500px'))
    play_btn = widgets.ToggleButton(value=False, description='▶ Play',
                                    button_style='success',
                                    layout=widgets.Layout(width='110px'))

    audio_w = None
    if _full_audio is not None:
        t0 = float(seg['time stamp'])
        s0  = int(t0 * _sr)
        s1  = int((t0 + float(seg['duration'])) * _sr)
        audio_w = _make_audio_html(_full_audio[s0:s1], _sr)

    _state = {'playing': False}

    def _show(idx):
        if 0 <= idx < len(all_frames):
            f = all_frames[idx]
            img_w.value = _frame_bytes(f, vrows[f])

    slider.observe(lambda c: _show(c['new']), names='value')

    def make_play_cb():
        def _cb(change):
            if change['new']:
                _state['playing'] = True
                play_btn.description = '⏸ Pause'
                start = slider.value
                def _run():
                    for i in range(start, len(all_frames)):
                        if not _state['playing']:
                            break
                        slider.value = i
                        _time.sleep(1.0)
                    if _state['playing']:
                        _state['playing'] = False
                        play_btn.value       = False
                        play_btn.description = '▶ Play'
                threading.Thread(target=_run, daemon=True).start()
            else:
                _state['playing']    = False
                play_btn.description = '▶ Play'
        return _cb

    play_btn.observe(make_play_cb(), names='value')
    _show(0)

    children = [header, img_w, widgets.HBox([play_btn, slider])]
    if audio_w is not None:
        children.append(audio_w)
    children.append(widgets.HTML("<hr style='margin:12px 0'>"))
    return widgets.VBox(children)

if not os.path.isdir(FRAMES_DIR):
    print(f"[!] '{FRAMES_DIR}/' not found.")
else:
    frame_lookup = data_visual.set_index('frame_number')
    cA, cB = extract_candidates(DEBATE)
    for speaker in [cA, cB, 'host']:
        top3 = debate_audio[debate_audio['speaker'] == speaker].nlargest(3, 'duration')
        for rank, (_, seg) in enumerate(top3.iterrows(), start=1):
            frames = sorted([f for f in seg['frames'] if f in frame_lookup.index])
            if frames:
                ipy_display(_make_block(seg, frames, rank, speaker, frame_lookup))


WAV loaded: sr=44100 Hz, duration=2048s
